In [1]:
!pip install /kaggle/input/rdkit-2025-3-3-cp311/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl
!pip install mordred --no-index --find-links=file:///kaggle/input/mordred-1-2-0-py3-none-any/

Processing /kaggle/input/rdkit-2025-3-3-cp311/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl
Looking in links: file:///kaggle/input/mordred-1-2-0-py3-none-any/
Processing /kaggle/input/mordred-1-2-0-py3-none-any/mordred-1.2.0-py3-none-any.whl
Processing /kaggle/input/mordred-1-2-0-py3-none-any/networkx-2.8.8-py3-none-any.whl (from mordred)
  Attempting uninstall: networkx
    Found existing installation: networkx 3.5
    Uninstalling networkx-3.5:
      Successfully uninstalled networkx-3.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-image 0.25.2 requires networkx>=3.0, but you have networkx 2.8.8 which is incompatible.
nx-cugraph-cu12 25.2.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have

In [2]:
import warnings
warnings.simplefilter(action='ignore')

import os
import re
import requests
import joblib, numpy

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import scipy.stats as stats

from catboost import CatBoostRegressor, Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoLars
from sklearn.linear_model import Lasso

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import StackingRegressor

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

In [3]:
train = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/train.csv')
test = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/test.csv')

def improved_tokenize_smiles(smiles):
    pattern = r'(\[[^\[\]]{1,10}\]|Br|Cl|[bcnops]|[BCNOFPSI]|[()=+\-#@:/\\]|\d+|\.)'
    tokens = re.findall(pattern, smiles)
    reconstructed = ''.join(tokens)
    remaining = smiles.replace(reconstructed, '')
    if remaining:
        tokens.extend(list(remaining))
    
    return tokens

def create_smiles_features(train_smiles, test_smiles):
    train_tokens = [improved_tokenize_smiles(smiles) for smiles in train_smiles]
    test_tokens = [improved_tokenize_smiles(smiles) for smiles in test_smiles]
    all_tokens = []
    for tokens in train_tokens + test_tokens:
        all_tokens.extend(tokens)
    
    vocab = sorted(list(set(all_tokens)))
    vocab_dict = {token: idx for idx, token in enumerate(vocab)}
    
    print(f"Vocabulary size: {len(vocab)}")
    print(f"Max sequence length: {max([len(t) for t in train_tokens + test_tokens])}")
    
    def get_token_counts(token_list, vocab_dict):
        features = np.zeros(len(vocab_dict))
        for tokens in token_list:
            for token in tokens:
                if token in vocab_dict:
                    features[vocab_dict[token]] += 1
        return features
    
    train_counts = np.array([get_token_counts([tokens], vocab_dict) for tokens in train_tokens])
    test_counts = np.array([get_token_counts([tokens], vocab_dict) for tokens in test_tokens])
    
    max_len = max([len(t) for t in train_tokens + test_tokens])
    max_len = min(max_len, 500)
    
    def pad_sequence(tokens, max_len, vocab_dict):
        sequence = []
        for i, token in enumerate(tokens[:max_len]):
            if token in vocab_dict:
                sequence.append(vocab_dict[token])
            else:
                sequence.append(0)
        while len(sequence) < max_len:
            sequence.append(0)
        
        return sequence
    
    train_sequences = np.array([pad_sequence(tokens, max_len, vocab_dict) for tokens in train_tokens])
    test_sequences = np.array([pad_sequence(tokens, max_len, vocab_dict) for tokens in test_tokens])
    
    def get_chemical_descriptors(smiles):
        features = {}
        
        features['length'] = len(smiles)
        features['num_atoms'] = len(re.findall(r'[A-Z][a-z]?', smiles))
        features['num_bonds'] = smiles.count('=') + smiles.count('#')
        features['num_rings'] = smiles.count('1') + smiles.count('2') + smiles.count('3') + \
                               smiles.count('4') + smiles.count('5') + smiles.count('6')
        features['num_branches'] = smiles.count('(') + smiles.count(')')
        features['num_aromatic'] = smiles.count('[') + smiles.count(']')
        features['carbon_count'] = smiles.count('C') + smiles.count('c')
        features['nitrogen_count'] = smiles.count('N') + smiles.count('n')
        features['oxygen_count'] = smiles.count('O') + smiles.count('o')
        features['sulfur_count'] = smiles.count('S') + smiles.count('s')
        features['phosphorus_count'] = smiles.count('P') + smiles.count('p')
        features['fluorine_count'] = smiles.count('F')
        features['chlorine_count'] = smiles.count('Cl')
        features['bromine_count'] = smiles.count('Br')
        features['alcohol_groups'] = smiles.count('OH')
        features['carboxyl_groups'] = smiles.count('COOH')
        features['ester_groups'] = smiles.count('COO')
        features['ether_groups'] = smiles.count('O') - features['alcohol_groups'] - \
                                  features['carboxyl_groups'] - features['ester_groups']
        features['complexity_score'] = features['num_rings'] + features['num_branches'] + \
                                     features['num_bonds'] * 0.5
        total_atoms = max(features['num_atoms'], 1)
        features['heteroatom_ratio'] = (features['nitrogen_count'] + features['oxygen_count'] + 
                                      features['sulfur_count'] + features['phosphorus_count']) / total_atoms
        features['carbon_ratio'] = features['carbon_count'] / total_atoms
        return features
    
    train_descriptors = [get_chemical_descriptors(smiles) for smiles in train_smiles]
    test_descriptors = [get_chemical_descriptors(smiles) for smiles in test_smiles]
    descriptor_names = list(train_descriptors[0].keys())
    train_desc_df = pd.DataFrame(train_descriptors, columns=descriptor_names)
    test_desc_df = pd.DataFrame(test_descriptors, columns=descriptor_names)
    
    return {
        'vocab': vocab_dict,
        'train_counts': train_counts,
        'test_counts': test_counts,
        'train_sequences': train_sequences,
        'test_sequences': test_sequences,
        'train_descriptors': train_desc_df,
        'test_descriptors': test_desc_df
    }

In [4]:
smiles_features = create_smiles_features(train['SMILES'], test['SMILES'])

Vocabulary size: 100
Max sequence length: 610


In [5]:
def get_ngram_features(tokens, top_ngrams, n=2):
    features = np.zeros(len(top_ngrams))
    for i in range(len(tokens) - n + 1):
        ngram = tuple(tokens[i:i+n])
        if ngram in top_ngrams:
            features[top_ngrams.index(ngram)] += 1
    return features

def create_ngram_features(token_list, n=2, top_k=1000):
    from collections import Counter
    
    all_ngrams = []
    for tokens in token_list:
        for i in range(len(tokens) - n + 1):
            ngram = tuple(tokens[i:i+n])
            all_ngrams.append(ngram)
    
    ngram_counts = Counter(all_ngrams)
    top_ngrams = [ngram for ngram, _ in ngram_counts.most_common(top_k)]
    
    def get_ngram_features(tokens, top_ngrams):
        features = np.zeros(len(top_ngrams))
        for i in range(len(tokens) - n + 1):
            ngram = tuple(tokens[i:i+n])
            if ngram in top_ngrams:
                features[top_ngrams.index(ngram)] += 1
        return features
    
    return top_ngrams, [get_ngram_features(tokens, top_ngrams) for tokens in token_list]

In [6]:
train_tokens_list = [improved_tokenize_smiles(smiles) for smiles in train['SMILES']]
test_tokens_list = [improved_tokenize_smiles(smiles) for smiles in test['SMILES']]

top_bigrams, train_bigram_features = create_ngram_features(train_tokens_list + test_tokens_list, n=2, top_k=500)
_, test_bigram_features = create_ngram_features(test_tokens_list, n=2, top_k=500)
test_bigram_features = [get_ngram_features(tokens, top_bigrams) for tokens in test_tokens_list]

train_bigram_features = [get_ngram_features(tokens, top_bigrams, 2) for tokens in train_tokens_list]
test_bigram_features = [get_ngram_features(tokens, top_bigrams, 2) for tokens in test_tokens_list]

train_bigram_df = pd.DataFrame(train_bigram_features, columns=[f'bigram_{i}' for i in range(len(top_bigrams))])
test_bigram_df = pd.DataFrame(test_bigram_features, columns=[f'bigram_{i}' for i in range(len(top_bigrams))])

print("Combining features...")

Combining features...


In [7]:
train_base = train.drop(columns=['SMILES'] if 'SMILES' in train.columns else [])
test_base = test.drop(columns=['SMILES'] if 'SMILES' in test.columns else [])

train_counts_df = pd.DataFrame(smiles_features['train_counts'], 
                              columns=[f'token_count_{i}' for i in range(smiles_features['train_counts'].shape[1])])
test_counts_df = pd.DataFrame(smiles_features['test_counts'], 
                             columns=[f'token_count_{i}' for i in range(smiles_features['test_counts'].shape[1])])

train_bigram_df = pd.DataFrame(train_bigram_features, columns=[f'bigram_{i}' for i in range(len(top_bigrams))])
test_bigram_df = pd.DataFrame(test_bigram_features, columns=[f'bigram_{i}' for i in range(len(top_bigrams))])

train_final = pd.concat([
    train_base,
    smiles_features['train_descriptors'],
    train_counts_df,
    train_bigram_df
], axis=1)

test_final = pd.concat([
    test_base,
    smiles_features['test_descriptors'], 
    test_counts_df,
    test_bigram_df
], axis=1)

print(f"Final feature count: {train_final.shape[1]}")

Final feature count: 627


In [8]:
train_split, val_split = train_test_split(
    train_final, test_size=0.07, random_state=42
)

train_split_Tg = train_split.drop(columns = ["FFV", "Tc", "Density", "Rg"]).dropna()
train_split_FFV = train_split.drop(columns = ["Tg", "Tc", "Density", "Rg"]).dropna()
train_split_Tc = train_split.drop(columns = ["Tg", "FFV", "Density", "Rg"]).dropna()
train_split_Density = train_split.drop(columns = ["Tg", "FFV", "Tc", "Rg"]).dropna()
train_split_Rg = train_split.drop(columns = ["Tg", "FFV", "Tc", "Density"]).dropna()

val_split_Tg = train_split.drop(columns = ["FFV", "Tc", "Density", "Rg"]).dropna()
val_split_FFV = train_split.drop(columns = ["Tg", "Tc", "Density", "Rg"]).dropna()
val_split_Tc = train_split.drop(columns = ["Tg", "FFV", "Density", "Rg"]).dropna()
val_split_Density = train_split.drop(columns = ["Tg", "FFV", "Tc", "Rg"]).dropna()
val_split_Rg = train_split.drop(columns = ["Tg", "FFV", "Tc", "Density"]).dropna()

In [9]:
improved_catboost_params = {
    'iterations': 5000,
    'learning_rate': 0.005,
    'depth': 6,
    'l2_leaf_reg': 3.0,
    'min_child_samples': 50,
    'colsample_bylevel': 0.8,
    'od_wait': 100,
    'random_state': 42,
    'od_type': 'Iter',
    'bootstrap_type': 'Bayesian',
    'grow_policy': 'Depthwise',
    'logging_level': 'Silent',
    'loss_function': 'MultiRMSE'
}

R_Forest_parm = {'n_estimators' : 25, 
                 'min_samples_split' : 2, 
                 'max_depth' : 10, 
                 'min_samples_leaf' : 2, 
                 'random_state' : 42}
        
Extra_parm = {'n_estimators' : 50, 
              'min_samples_split' : 2, 
              'max_depth' : 8, 
              'min_samples_leaf' : 2, 
              'random_state' : 42}
        
LGBM_R_parm = {'colsample_bytree': 0.6657998165699125,
               'drop_rate': 0.8303473371870002,
               'learning_rate': 0.2762739125344964,
               'max_bin': 9983,
               'max_depth': 8623,
               'max_drop': 5480,
               'min_child_samples': 101,
               'min_data_in_leaf': 426,
               'n_estimators': 1628,
               'num_leaves': 3640,
               'objective': 'regression_l1',
               'reg_alpha': 0.39940189926691194,
               'reg_lambda': 0.9567353299218986,
               'skip_drop': 0.6274640597528743,
               'verbosity': -1}
        
XGB_R_parm = {'colsample_bytree' : 0.871893537724492,
              'gamma' : 1,
              'learning_rate' : 0.923192518624813,
              'max_depth' : 15,
              'min_child_weight' : 1,
              'n_estimators' : 17748,
              'reg_alpha' : 45,
              'reg_lambda' : 0.8598696247943665,
              'subsample' : 0.9109367356405415,
              'random_state' : 42}

element_pre = {}

for train_split, val_split, var in [(train_split_Tg, val_split_Tg, 'Tg'),
                                    (train_split_FFV, val_split_FFV, 'FFV'),
                                    (train_split_Tc, val_split_Tc, 'Tc'),
                                    (train_split_Density, val_split_Density, 'Density'),
                                    (train_split_Rg, val_split_Rg, 'Rg')]:

    print(f"Training CatBoost {var} model...")
    CatBoost = CatBoostRegressor(**improved_catboost_params)
    
    print(f"Training XGBoost {var} model...")
    XGBoost = XGBRegressor(**XGB_R_parm)
    
    print(f"Training LGBM {var} model...")
    LGBM = LGBMRegressor(**LGBM_R_parm)
    
    print(f"Training RandomForest {var} model...")
    RandomForest = RandomForestRegressor(**R_Forest_parm)
    
    print(f"Training ExtraTrees {var} model...")
    ExtraTrees = ExtraTreesRegressor(**Extra_parm)

    estimators = [('CatBoost', CatBoost), 
                  ('RandomForest', RandomForest), ('ExtraTrees', ExtraTrees)]
    
    model = StackingRegressor(estimators, final_estimator = RidgeCV())
    model.fit(train_split.drop(columns = [var]), train_split[var])

    element_pre[var] = model.predict(test_final)

Training CatBoost Tg model...
Training XGBoost Tg model...
Training LGBM Tg model...
Training RandomForest Tg model...
Training ExtraTrees Tg model...
Training CatBoost FFV model...
Training XGBoost FFV model...
Training LGBM FFV model...
Training RandomForest FFV model...
Training ExtraTrees FFV model...
Training CatBoost Tc model...
Training XGBoost Tc model...
Training LGBM Tc model...
Training RandomForest Tc model...
Training ExtraTrees Tc model...
Training CatBoost Density model...
Training XGBoost Density model...
Training LGBM Density model...
Training RandomForest Density model...
Training ExtraTrees Density model...
Training CatBoost Rg model...
Training XGBoost Rg model...
Training LGBM Rg model...
Training RandomForest Rg model...
Training ExtraTrees Rg model...


In [10]:
# Make predictions
print("Making predictions...")

test_predictions_Tg = element_pre['Tg']
test_predictions_FFV = element_pre['FFV']
test_predictions_Tc = element_pre['Tc']
test_predictions_Density = element_pre['Density']
test_predictions_Rg = element_pre['Rg']

# Create submission
submission = pd.DataFrame({
    'id': test_final['id'],
    'Tg': test_predictions_Tg,
    'FFV': test_predictions_FFV, 
    'Tc': test_predictions_Tc,
    'Density': test_predictions_Density,
    'Rg': test_predictions_Rg
})

print("Feature importance (top 20):")
print("Submission file saved as 'improved_submission.csv'")

Making predictions...
Feature importance (top 20):
Submission file saved as 'improved_submission.csv'


In [11]:
submission.to_csv("submission.csv", index=False)
submission

,id,Tg,FFV,Tc,Density,Rg
0,1109053969,155.961515,0.374983,0.216615,1.118168,21.230077
1,1422188626,200.744258,0.379229,0.262107,1.096915,21.431831
2,2032016830,86.527104,0.356052,0.273337,1.156941,22.385267
